# Métodos para procesar valores faltantes
En los siguientes bloques se prueban diferentes métodos para manejar los valores faltantes:

- Eliminación de filas con valores faltantes.
- Sustitución con la media.
- Interpolación con valores cercanos.
- Imputación mediante otros algoritmos.

En cada método, se realiza lo siguiente:
- Se aplica la técnica de imputación correspondiente a los datos de entrenamiento, y test cuando corresponda.
- Se evalúa el rendimiento de un modelo de regresión lineal con los datos modificados.

El cuadernillo demuestra una aplicación práctica de estos métodos en un conjunto de datos y proporciona una comparación  de su rendimiento.

 Este tipo de análisis es crucial en el preprocesamiento de datos para el aprendizaje automático y la modelización estadística, donde la calidad y la integridad de los datos pueden influir significativamente en los resultados del modelo.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


In [2]:
# Definimos la URL del conjunto de datos
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"

# Especificamos los nombres de las columnas
column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration', 'Model Year', 'Origin', 'Car Name']

# Cargamos los datos en un DataFrame de pandas
data = pd.read_csv(url, sep=r'\s+', names=column_names, na_values='?')

# Eliminamos las columna 'Car Name' y 'Origin' ya que no es relevante para nuestro análisis
data = data.drop('Car Name', axis=1)

data = data.drop('Origin', axis=1)

# Mostramos las primeras filas del DataFrame
data

,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year
0,18.0,8,307.0,130.0,3504.0,12.0,70
1,15.0,8,350.0,165.0,3693.0,11.5,70
2,18.0,8,318.0,150.0,3436.0,11.0,70
3,16.0,8,304.0,150.0,3433.0,12.0,70
4,17.0,8,302.0,140.0,3449.0,10.5,70
...,...,...,...,...,...,...,...
393,27.0,4,140.0,86.0,2790.0,15.6,82
394,44.0,4,97.0,52.0,2130.0,24.6,82
395,32.0,4,135.0,84.0,2295.0,11.6,82
396,28.0,4,120.0,79.0,2625.0,18.6,82


In [ ]:
#Colocamos la variable objetivo al final
data = data[['Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration', 'Model Year', 'MPG']]


In [3]:
predictors = ['Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration', 'Model Year']

## Dividimos en Entrenamiento y Test



In [5]:
X = data[predictors]
y = data['MPG']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state = 0)

In [ ]:
# Entrenamos un modelo de regresión lineal
# Separamos en train y test

try:
  model = LinearRegression()
  model.fit(X_train, y_train)
except Exception as e:
  print(f"Error al entrenar el modelo: {e}")

Error al entrenar el modelo: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values


El modelo no funciona si existen datos faltantes.

## Gestión de Nans

En primer lugar, vamos a introducir más datos faltantes para un toque más realista.

In [ ]:
def insert_random_nans(df, percentage, seed=42):
    """
    Inserta valores NaN de manera aleatoria en un DataFrame.

    Parámetros:
    df (pandas.DataFrame): DataFrame original.
    percentage (float): Porcentaje de valores a convertir en NaN.
    seed (int): Semilla para la generación de números aleatorios (por defecto 42).

    Devuelve:
    pandas.DataFrame: Nuevo DataFrame con NaNs insertados.
    """
    # Crear una copia del DataFrame para evitar modificar el original
    df_with_nans = df.copy()

    # Calcular el número total de elementos en el DataFrame
    total_elements = df_with_nans.size

    # Calcular el número exacto de NaNs a insertar basándose en el porcentaje dado
    nans_to_insert = int(total_elements * percentage)

    # Establecer la semilla para asegurar la reproducibilidad en la generación de números aleatorios
    np.random.seed(0)

    # Iterar para insertar la cantidad calculada de NaNs
    for _ in range(nans_to_insert):
        # Seleccionar una fila al azar dentro del rango del DataFrame
        random_row = np.random.randint(0, len(df_with_nans))
        # Seleccionar una columna al azar dentro del rango de columnas del DataFrame
        random_column = np.random.randint(0, df_with_nans.shape[1])
        # Asignar el valor NaN en la posición aleatoria determinada por fila y columna
        df_with_nans.iat[random_row, random_column] = np.nan

    # Devolver el nuevo DataFrame con NaNs insertados
    return df_with_nans


In [26]:
# Reconstruimos los conjuntos completos originales para aplicar más NaNs
X_train['MPG'] = y_train # añade una nueva colmna
train_set = X_train

X_test['MPG'] = y_test # añade una nueva colmna
test_set = X_test

In [27]:
# Añadimos NaNs en el conjunto de entrenamiento (En las variables predictoras
# y en la variable objetivo)
train_set_nans = insert_random_nans(X_train,0.1)

# Analizamos el contenido con .info()
train_set_nans.info()

<class 'pandas.core.frame.DataFrame'>
Index: 318 entries, 64 to 172
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Cylinders     283 non-null    float64
 1   Displacement  284 non-null    float64
 2   Horsepower    293 non-null    float64
 3   Weight        277 non-null    float64
 4   Acceleration  289 non-null    float64
 5   Model Year    288 non-null    float64
 6   MPG           295 non-null    float64
dtypes: float64(7)
memory usage: 19.9 KB


In [28]:
# Añadimos NaNs en el conjunto de entrenamiento (En las variables predictoras
# y en la variable objetivo)
test_set_nans = insert_random_nans(X_test,0.1)

# Analizamos el contenido con .info()
test_set_nans.info()

<class 'pandas.core.frame.DataFrame'>
Index: 80 entries, 65 to 124
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Cylinders     71 non-null     float64
 1   Displacement  71 non-null     float64
 2   Horsepower    71 non-null     float64
 3   Weight        76 non-null     float64
 4   Acceleration  68 non-null     float64
 5   Model Year    76 non-null     float64
 6   MPG           72 non-null     float64
dtypes: float64(7)
memory usage: 5.0 KB


Comprobamos que la variable objetivo 'MPG' contiene todos los valores.


## Eliminamos filas con valores faltantes

In [29]:
# Eliminamos las filas del set de entrenamiento con valores faltantes
train_set_removing_nans = train_set_nans.dropna()

# Puede tener Nans del dataset original
test_set_removing_nans = test_set_nans.dropna()

In [30]:
train_set_removing_nans.info()

<class 'pandas.core.frame.DataFrame'>
Index: 153 entries, 64 to 47
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Cylinders     153 non-null    float64
 1   Displacement  153 non-null    float64
 2   Horsepower    153 non-null    float64
 3   Weight        153 non-null    float64
 4   Acceleration  153 non-null    float64
 5   Model Year    153 non-null    float64
 6   MPG           153 non-null    float64
dtypes: float64(7)
memory usage: 9.6 KB


In [31]:
test_set_removing_nans.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39 entries, 74 to 235
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Cylinders     39 non-null     float64
 1   Displacement  39 non-null     float64
 2   Horsepower    39 non-null     float64
 3   Weight        39 non-null     float64
 4   Acceleration  39 non-null     float64
 5   Model Year    39 non-null     float64
 6   MPG           39 non-null     float64
dtypes: float64(7)
memory usage: 2.4 KB


In [32]:
# Separar en características y variable objetivo
X_train = train_set_removing_nans[predictors]
y_train = train_set_removing_nans['MPG']

# Entrenamos un modelo de regresión lineal
model = LinearRegression()
model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [33]:
# Evaluamos sobre el conjunto de test completo
X_test = test_set_removing_nans[predictors]
y_test = test_set_removing_nans['MPG']


# Calcular el MSE con los datos de prueba
y_pred= model.predict(X_test)
mse_removing_nans= mean_squared_error(y_test, y_pred)
print(f"MSE después de eliminar filas con NaNs: %.2f " % mse_removing_nans)

MSE después de eliminar filas con NaNs: 8.27 


## Sustituimos los valores faltantes con la media

https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html#pandas.DataFrame.fillna

In [40]:
# Sustituir NaNs con la media de cada columna
train_set_filled_w_mean = train_set_nans.fillna(train_set_nans.mean())
test_set_filled_w_mean = test_set_nans.fillna(test_set_nans.mean())

# Esta línea crea un nuevo dataframe, train_set_filled_w_mean, que es una copia de train_set_nans
# donde todos los valores NaN han sido reemplazados por la media de sus respectivas columnas.

In [41]:
train_set_nans.mean()

Cylinders          5.420495
Displacement     193.420775
Horsepower       103.515358
Weight          2973.732852
Acceleration      15.675087
Model Year        76.173611
MPG               23.695932
dtype: float64

In [43]:
train_set_filled_w_mean.head(10)

,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,MPG
64,8.000000,318.000000,150.0,4135.000000,13.500000,72.000000,15.0
55,4.000000,97.000000,60.0,1834.000000,19.000000,71.000000,27.0
317,4.000000,97.000000,78.0,2188.000000,15.800000,76.173611,34.3
102,4.000000,97.000000,46.0,1950.000000,21.000000,73.000000,26.0
358,4.000000,120.000000,74.0,2635.000000,15.675087,81.000000,31.6
331,5.420495,97.000000,67.0,2145.000000,18.000000,80.000000,33.8
371,4.000000,135.000000,84.0,2525.000000,16.000000,82.000000,29.0
261,6.000000,193.420775,120.0,3410.000000,15.100000,78.000000,18.1
299,4.000000,141.000000,71.0,3190.000000,24.800000,79.000000,27.2
378,4.000000,105.000000,63.0,2973.732852,14.700000,82.000000,38.0


In [44]:
train_set_nans.head(10)

,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,MPG
64,8.0,318.0,150.0,4135.0,13.5,72.0,15.0
55,4.0,97.0,60.0,1834.0,19.0,71.0,27.0
317,4.0,97.0,78.0,2188.0,15.8,NaN,34.3
102,4.0,97.0,46.0,1950.0,21.0,73.0,26.0
358,4.0,120.0,74.0,2635.0,NaN,81.0,31.6
331,NaN,97.0,67.0,2145.0,18.0,80.0,33.8
371,4.0,135.0,84.0,2525.0,16.0,82.0,29.0
261,6.0,NaN,120.0,3410.0,15.1,78.0,18.1
299,4.0,141.0,71.0,3190.0,24.8,79.0,27.2
378,4.0,105.0,63.0,NaN,14.7,82.0,38.0


In [45]:
# Separamos en características y variable objetivo
X_train = train_set_filled_w_mean[predictors]
y_train = train_set_filled_w_mean['MPG']

# Entrenamos un modelo de regresión lineal
model = LinearRegression()
model.fit(X_train, y_train)


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


Para evaluar el modelo en el conjunto de test tendremos dos opciones:



*   Eliminamos las filas con valores faltantes: En este caso es estamos evaluando el modelo en un conjunto de datos de tamaño más pequeño.

*   Sustituimos los valores faltantes con la media: En este caso disponemos de más datos, pero podemos perder algo de fiabilidad en los mismos por las sustituciones.

En un caso real podemos encontrarnos una situación en la que sólo tenemos una observación con datos faltantes con la cual tenemos que hacer una predicción, en cuyo caso sólo nos quedaría buscar algún método de sustitución, lo que corresponde al segundo caso.

Sólo si nos pudiéramos permitir 'dejar' de hacer predicciones para ciertas observaciones, tendría sentido la eliminación de filas en el conjunto de datos, llamémosle, de 'producción'. Aunque no es un caso muy realista, realizaremos el cálculo a modo de demostración.


In [47]:
# Evaluamos sobre el conjunto de test completo
X_test = test_set_filled_w_mean[predictors]
y_test = test_set_filled_w_mean['MPG']

# Calcular el MSE con los datos de test
y_pred= model.predict(X_test)
mse_filled_w_mean= mean_squared_error(y_test, y_pred)

print(f"MSE después de rellenar NaNs con la media: %.2f " % mse_filled_w_mean)

MSE después de rellenar NaNs con la media: 23.00 


Al eliminar las filas del conjunto de test, estamos evaluando el modelos con un conjunto de test más pequeño. No tiene que dar necesariamente un resultado mejor, pero, en cualquier caso, el resultado será menos fiable pues se evalúa sobre una cantidad de datos menor.

En este caso, tenemos más filas para evaluar al modelo. Por ello, este resultado no es comparable con el método anterior de eliminar las filas. Si acaso, el resultado es más fiable, y responde a un caso más realista.

## Interpolación de Datos

La interpolación puede tener más sentido frente a la imputación por la media cuando los datos tienen algún ordenamiento lógico, como es el caso de las series temporales.

En este caso nuestros datos no parecen tener ese orden, pero practiquemos en todo caso la interpolación.

https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.interpolate.html

In [48]:
# Sustituir NaNs con la media de cada columna
train_set_interpolated = train_set_nans.interpolate(method='linear', limit_direction='both')
test_set_interpolated = test_set_nans.interpolate(method='linear', limit_direction='both')

In [6]:
# Separamos en características y variable objetivo
X_train = train_set_interpolated[predictors]
y_train = train_set_interpolated['MPG']

# Entrenamos un modelo de regresión lineal
model = LinearRegression()
model.fit(X_train, y_train)


NameError: name 'train_set_interpolated' is not defined

In [50]:
# Evaluamos sobre el conjunto de test completo
X_test = test_set_interpolated[predictors]
y_test = test_set_interpolated['MPG']

# Calcular el MSE con los datos de prueba
y_pred= model.predict(X_test)
mse= mean_squared_error(y_test, y_pred)
print(f"MSE después de rellenar NaNs mediante interpolación también en el conjunto de test: %.2f " % mse)

MSE después de rellenar NaNs mediante interpolación también en el conjunto de test: 26.03 


El resultado es coherente con nuestra estructura de los datos, que no tienen ningún ordenamiento lógico, por lo que la interpolación tiene menos sentido. Así, vemos que esta técnica es menos efectiva en este caso que la sustitución por la media.

## Imputación basada en primeros vecinos

https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html

In [ ]:
from sklearn.impute import KNNImputer

# Imputación KNN
imputer = KNNImputer(n_neighbors=5)
train_set_imputed_knn = imputer.fit_transform(train_set_nans)
test_set_imputed_knn = imputer.fit_transform(test_set_nans)

In [52]:
# En este caso obtenemos un array y no un dataframe
train_set_imputed_knn

array([[  8.  , 318.  , 150.  , ...,  13.5 ,  72.  ,  15.  ],
       [  4.  ,  97.  ,  60.  , ...,  19.  ,  71.  ,  27.  ],
       [  4.  ,  97.  ,  78.  , ...,  15.8 ,  77.  ,  34.3 ],
       ...,
       [  4.  ,  68.  ,  49.  , ...,  19.5 ,  73.  ,  29.  ],
       [  6.  , 250.  , 100.  , ...,  15.  ,  71.  ,  19.  ],
       [  4.  ,  90.  ,  71.  , ...,  16.5 ,  75.  ,  31.02]],
      shape=(318, 7))

In [53]:
# Convertimos el array resultante en un DataFrame
train_set_imputed_knn = pd.DataFrame(train_set_imputed_knn, columns=train_set_nans.columns, index=train_set_nans.index)
test_set_imputed_knn = pd.DataFrame(test_set_imputed_knn, columns=test_set_nans.columns, index=test_set_nans.index)

In [54]:
# Separar en características y variable objetivo
X_train = train_set_imputed_knn[predictors]
y_train = train_set_imputed_knn['MPG']

# Entrenamos un modelo de regresión lineal
model = LinearRegression()
model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [55]:
# Visualizamos el conjunto de test
test_set

,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,MPG
65,8,351.0,153.0,4129.0,13.0,72,14.0
132,4,140.0,75.0,2542.0,17.0,74,25.0
74,8,302.0,140.0,4294.0,16.0,72,13.0
78,4,120.0,87.0,2979.0,19.5,72,21.0
37,6,232.0,100.0,3288.0,15.5,71,18.0
...,...,...,...,...,...,...,...
269,4,105.0,75.0,2230.0,14.5,78,30.9
235,4,97.0,75.0,2265.0,18.2,77,26.0
145,4,83.0,61.0,2003.0,19.0,74,32.0
196,4,98.0,60.0,2164.0,22.1,76,24.5


In [56]:
# Rellenamos el conjunto de test mediante imputación KNN (en y_test no hay faltantes)
imputer_target = KNNImputer(n_neighbors=5)
test_set_imputed_knn = imputer_target.fit_transform(test_set)

In [57]:
# Convertimos el array resultante en un DataFrame
test_set_imputed_knn = pd.DataFrame(test_set_imputed_knn, columns=test_set.columns, index=test_set.index)

# Ahora 'test_set_imputed_knn' es un DataFrame con los valores imputados
test_set_imputed_knn

,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,MPG
65,8.0,351.0,153.0,4129.0,13.0,72.0,14.0
132,4.0,140.0,75.0,2542.0,17.0,74.0,25.0
74,8.0,302.0,140.0,4294.0,16.0,72.0,13.0
78,4.0,120.0,87.0,2979.0,19.5,72.0,21.0
37,6.0,232.0,100.0,3288.0,15.5,71.0,18.0
...,...,...,...,...,...,...,...
269,4.0,105.0,75.0,2230.0,14.5,78.0,30.9
235,4.0,97.0,75.0,2265.0,18.2,77.0,26.0
145,4.0,83.0,61.0,2003.0,19.0,74.0,32.0
196,4.0,98.0,60.0,2164.0,22.1,76.0,24.5


In [58]:
# Llevamos a cabo la evaluación
X_test = test_set_imputed_knn[predictors]
y_test = test_set_imputed_knn['MPG']

# Calcular el MSE con los datos de prueba
y_pred= model.predict(X_test)
mse_knn= mean_squared_error(y_test, y_pred)
___
print(f"MSE después de rellenar NaNs mediante imputación KNN: %.2f " % mse_knn)

MSE después de rellenar NaNs mediante imputación KNN: 12.67 
